### Задание со звездочкой 17. Dunn index
Реализуйте Dunn Index. Будет засчитываться не абы-какая реализация, а соответствующая по стилю и оформлению реализации метрик в sklearn: обязательны докстринги, валидаторы и все такое.

То, как реализованы другие метрики, можно посмотреть [тут](https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/metrics/cluster/_unsupervised.py#L195).

In [1]:
import numpy as np

from sklearn.metrics import pairwise_distances
from sklearn.metrics.pairwise import _VALID_METRICS
from sklearn.utils._param_validation import validate_params, StrOptions
from sklearn.utils.validation import check_array, check_consistent_length

@validate_params(
    {
        "X": ["array-like", "sparse matrix"],
        "labels": ["array-like"],
        "metric": [StrOptions(set(_VALID_METRICS) | {"precomputed"}), callable],
    },
    prefer_skip_nested_validation=True,
)
def dunn_score(X, labels, *, metric = "euclidean"):
    """Compute the Dunn index of a cluster partition.

    The Dunn index is defined as the ratio between the smallest
    inter-cluster distance and the largest intra-cluster distance
    (cluster diameter). The inter-cluster distance between two
    clusters is measured by the distance between their closest points.
    The intra-cluster distance is the largest distance betwen two
    distinct points within one cluster.

    Parameters
    ----------
    X : {array-like, sparse matrix} of shape (n_samples_X, n_samples_X) or (n_samples_X, n_features)
        Array of pairwise distances between samples, or a feature array.
        The shape of the array should be (n_samples_X, n_samples_X) if
        metric == "precomputed" and (n_samples_X, n_features) otherwise.

    labels : array-like of shape (n_samples,)
        Predicted cluster labels for each sample.

    metric : str or callable, default='euclidean'
        The metric to use when calculating distance between instances in a
        feature array. If metric is a string, it must be one of the options
        allowed by :func:`~sklearn.metrics.pairwise_distances`. If ``X`` is
        the distance array itself, use ``metric="precomputed"``.

    Returns
    -------
    score : float
        Dunn index. Higher values indicate better and more compact clustering.

    Notes
    -----
    The Dunn index is only defined when there are at least two
    clusters and at least two samples overall. If all clusters have
    zero diameter (e.g. all points are duplicated), the function
    returns ``np.inf``.
    
    References
    ----------

    .. [1] `Dunn J. C. (1974). "Well-Separated Clusters and Optimal
        Fuzzy Partitions." Journal of Cybernetics, 4(1), 95–104.
        <https://doi.org/10.1080/01969727408546059>`
        
    Examples
    --------
    >>> data = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
    >>> y_pred = np.array([0, 0, 1, 1, 1])
    >>> dunn_score(data, y_pred)
    0.5
    """
    X = check_array(X, accept_sparse=["csr", "csc"], estimator="dunn_score")
    labels = np.asarray(labels)
    check_consistent_length(X, labels)
    n_clusters = len(np.unique(labels))

    if X.shape[0] < 2:
        raise ValueError("Dunn index is not defined for less than 2 samples.")
    if n_clusters < 2:
        raise ValueError(
            "Dunn index is not defined for less than 2 clusters. "
            f"Got {n_clusters} cluster."
        )
    if n_clusters == X.shape[0]:
        raise ValueError("Dunn index is not defined when every sample is its own cluster.")

    is_same_cluster = (labels == labels[:, None])
    distances = pairwise_distances(X, metric = metric)

    dmin = np.min(distances, initial = np.inf, where = ~is_same_cluster)
    dmax = np.max(distances, initial = -np.inf, where = is_same_cluster)

    if dmax == 0:
        return np.inf
    return dmin / dmax

In [2]:
data = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
y_pred = np.array([0, 0, 1, 1, 1])
dunn_score(data, y_pred)

0.5

In [3]:
# Example from https://permetrics.readthedocs.io/en/joss-paper/pages/clustering/DI.html for reference

from permetrics import ClusteringMetric
print(ClusteringMetric(X=data, y_pred=y_pred).dunn_index(use_modified = False))

0.5


### Задание со звездочкой 18. Кластеризация текстов

Мы сделали кластеризацию **слов** на основе их векторных представлений. На самом деле, кластеризовывать тексты гораздо сложнее и интереснее; для этой задачи существуют специальные модели, эти модели обучаются на **корпусе текстов**. Мы поговорим о таких моделях в следующем семестре, и быстро обобщим их до **тематического моделирования**, но это будет потом. А что делать, если датасета с текстами нет? Наивный подход состоит в том, чтобы как-нибудь аггрегировать векторные представления слов, например взять их среднее.

Ваша задача - предложите способ кластеризации **предложений**, который бы использовал только векторные представления слов, реализуйте и продемонстрируйте, что ваш способ работает лучше, чем наивный. 

**Подсказка:** _того, что было рассказано сегодня, должно быть достаточно, нужно только грамотно скомпоновать разные части практики._

In [4]:
# Сгенерировано Alice AI LLM. Промпт:
# Сгенерируй 100 длинных предложений (от 10 слов) на темы:
# - королевская тематика
# - домашние животные
# - спорт
# - технологии
# 
# На каждую тему должно быть по 25 предложений.

sentences = [
    "Королева в роскошном платье с золотой вышивкой торжественно вошла в тронный зал, где уже собрались все придворные, чтобы услышать её важное объявление",
    "Наследный принц, несмотря на юный возраст, уже проявлял мудрость и хладнокровие, необходимые для управления огромным королевством в неспокойные времена",
    "Церемония коронации, проходившая в древнем соборе, поражала великолепием: тысячи свечей, хоры певчих и сияющая корона, передаваемая из поколения в поколение",
    "Королевский двор жил по строгим правилам этикета, где каждый жест, взгляд и слово имели своё особое значение и могли повлиять на судьбу придворного",
    "В тайной комнате замка хранились древние свитки, повествующие о магических договорах, заключённых первыми королями с духами лесов и гор",
    "Принцесса, воспитанная в уединении, мечтала не о балах, а о путешествиях в далёкие страны, где она могла бы изучать языки и обычаи других народов",
    "Король, понимая, что война неизбежна, приказал укрепить границы и собрать совет из самых опытных военачальников для разработки стратегии обороны",
    "На ежегодном королевском балу гости появлялись в нарядах, стоимость которых могла сравниться с годовым доходом небольшого города, но главное — с безупречным вкусом",
    "Дворецкий, прослуживший при дворе более сорока лет, знал все секреты замка, включая потайные ходы, о которых не догадывались даже члены королевской семьи",
    "Герб королевства, украшенный изображением льва и трёх звёзд, символизировал силу, мудрость и вечную связь с небесными покровителями",
    "Послы из соседних государств прибыли с дарами и предложениями о союзе, но король не спешил давать ответ, взвешивая все возможные последствия",
    "В библиотеке замка хранились книги, написанные на языках, давно забытых в мире, и лишь несколько учёных могли их прочесть",
    "Принцесса, увлечённая ботаникой, создала в саду уникальный лабиринт из редких растений, где каждый поворот открывал новый аромат и цвет",
    "Король, несмотря на давление советников, отказался подписать мирный договор, считая его унизительным для чести своей страны",
    "На рассвете, когда первые лучи солнца коснулись шпилей замка, стража сменила караул, а в кухне уже начинали готовить завтрак для всей королевской семьи",
    "В сокровищнице, охраняемой рыцарями, лежали не только золото и драгоценности, но и реликвии, обладающие мистической силой",
    "Королева, обладая редким даром красноречия, могла убедить даже самых упрямых лордов поддержать её инициативы по улучшению жизни простых людей",
    "На турнире рыцарей, организованном в честь дня рождения принца, сражались лучшие воины королевства, демонстрируя невероятную силу и мастерство",
    "В старом крыле замка, куда редко заходили, хранились портреты предков, чьи глаза, казалось, следили за каждым, кто проходил мимо",
    "Король, осознавая, что его правление подходит к концу, начал готовить наследника, обучая его тонкостям дипломатии и военного искусства",
    "На празднике урожая весь город украсили гирляндами из цветов, а на главной площади устроили представление в честь королевской семьи",
    "Принцесса, тайно изучавшая магию, боялась, что её способности станут известны, ведь в королевстве колдовство было под запретом",
    "Совет лордов долго спорил о налогах, но королева предложила компромисс, который позволил сохранить доверие народа и пополнить казну",
    "В ночь полнолуния, согласно древнему обычаю, король должен был пройти испытание в лесу, чтобы доказать своё право на трон",
    "Дворец, построенный несколько веков назад, пережил войны, пожары и заговоры, но по-прежнему оставался символом незыблемости королевской власти.",
    "Мой пёс, радостно виляя хвостом, бросился навстречу, как только я открыл дверь, будто мы расстались не на час, а на целую неделю",
    "Кошка, уютно устроившись на подоконнике, наблюдала за птицами за окном, время от времени издавая тихое мурлыканье, полное предвкушения",
    "Попугай, выучив несколько фраз, с удовольствием повторял их в самый неподходящий момент, заставляя нас смеяться до слёз",
    "Щенок, впервые оказавшись в парке, с любопытством обнюхивал каждую травинку и пугался шума проезжающих велосипедов",
    "Кролик, живущий в просторной клетке, обожал свежие овощи и особенно морковку, которую он грыз с поразительной скоростью",
    "Хомяк, несмотря на маленькие размеры, проявил удивительную изобретательность, соорудив гнездо из кусочков бумаги и ваты",
    "Собака, обученная помогать людям с ограниченными возможностями, выполняла десятки команд и была настоящим членом семьи",
    "Кошка, обладающая независимым характером, позволяла гладить себя только тогда, когда сама этого хотела, и никогда — по принуждению",
    "Рыбки в аквариуме плавали в такт музыке, которую я включал по вечерам, создавая атмосферу умиротворения и покоя",
    "Морская свинка, услышав звук открывающейся дверцы, тут же подбегала к краю вольера, ожидая угощения и ласки",
    "Пёс, почувствовав, что хозяин расстроен, молча положил голову ему на колени, словно пытаясь передать свою поддержку",
    "Котёнок, играя с клубком ниток, запутался так сильно, что пришлось аккуратно распутывать его, не причиняя стресса",
    "Волнистый попугайчик, живя в паре, постоянно щебетал с партнёром, и их диалог напоминал весёлую беседу на непонятном языке",
    "Собака, привыкшая к режиму, каждое утро будила меня ровно в 7:00, деликатно трогая лапой за плечо",
    "Черепаха, медленно передвигаясь по террариуму, время от времени останавливалась, чтобы осмотреть окружающий мир своими внимательными глазами",
    "Кошка, заметив, что я беру фотоаппарат, тут же принимала грациозную позу, будто знала, что её фотографируют",
    "Щенок, ещё не умея контролировать свои эмоции, то радостно прыгал, то вдруг засыпал, свернувшись клубочком у батареи",
    "Канарейка, получив новую клетку, сначала настороженно осматривалась, но вскоре начала петь, наполняя дом мелодичными трелями",
    "Собака, выросшая в деревне, прекрасно понимала команды на местном диалекте, что удивляло приезжих гостей",
    "Кот, любивший высоту, забирался на самый верх шкафа и оттуда наблюдал за происходящим с видом властелина территории",
    "Морская свинка, обладая чутким слухом, реагировала на звук открывающегося пакета с кормом быстрее, чем я успевал его достать",
    "Попугай, подражая голосу хозяина, однажды ответил на телефонный звонок, чем вызвал настоящий переполох",
    "Кошка, привыкшая спать в кровати, всегда выбирала место у изголовья, где ей было теплее и удобнее",
    "Пёс, охраняя двор, внимательно следил за каждым прохожим, но никогда не лаял без причины, отличая друзей от незнакомцев",
    "Хомяк, активный по ночам, шуршал в своей клетке, устраивая перестановки, пока все в доме спали.",
    "Футбольная команда, проигрывая к перерыву со счётом 0:2, сумела собраться и забить три гола в последние двадцать минут, вырвав победу у сильного соперника",
    "Бегун, преодолевая последний километр марафона, чувствовал, как каждая мышца кричит от усталости, но воля к победе гнала его вперёд",
    "Тренер, анализируя ошибки предыдущего матча, разработал новую тактику, которая позволила команде играть более слаженно и агрессивно",
    "На олимпийском стадионе, заполненном до отказа, атмосфера была настолько напряжённой, что каждый вздох зрителей ощущался как часть общего ритма",
    "Баскетболист, сделав эффектный слэм-данк, вызвал бурю оваций, а его команда получила важный импульс для дальнейшего успеха",
    "Велосипедист, участвуя в многодневной гонке, ежедневно преодолевал сотни километров, сохраняя силы для финального рывка",
    "Теннисист, отбивая сложные подачи, демонстрировал невероятную реакцию и точность, заставляя соперника ошибаться раз за разом",
    "Команда по регби, несмотря на травмы и усталость, продолжала бороться до последней секунды, воплощая дух настоящего спортивного братства",
    "Плавец, установивший новый мировой рекорд, посвятил победу своим родителям, которые поддерживали его с первых тренировок",
    "На соревнованиях по гимнастике юная спортсменка выполнила сложную комбинацию без единой ошибки, заслужив высшие оценки судей",
    "Боксёр, пропустив сильный удар, сумел собраться и провести контратаку, которая принесла ему победу в решающем раунде",
    "Лыжник, спускаясь по сложному склону, контролировал каждое движение, чтобы избежать падения и сохранить лидерство в гонке",
    "Капитан футбольной сборной, подняв кубок над головой, поблагодарил болельщиков за неизменную поддержку и веру в команду",
    "В шахматном турнире гроссмейстеры часами обдумывали ходы, взвешивая каждую возможность и угрозу со стороны противника",
    "Команда по волейболу, проигрывая партию, сумела переломить ход игры благодаря мощной подаче и слаженной защите",
    "Марафонцы, стартовав в туманное утро, постепенно растягивались по трассе, каждый следуя своему темпу и стратегии",
    "Фигуристка, выполняя тройной прыжок, на мгновение замерла в воздухе, а затем безупречно приземлилась, вызвав аплодисменты зала",
    "Тренер по плаванию разработал индивидуальную программу тренировок, учитывающую особенности физиологии каждого спортсмена",
    "На турнире по боевым искусствам бойцы демонстрировали не только силу, но и дисциплину, уважение к сопернику и мастерство техники",
    "Футболист, получивший травму, всё же вышел на поле в решающем матче, показав пример самоотверженности и преданности команде",
    "В парусной регате капитаны умело использовали ветер и течение, чтобы обойти соперников и первыми достичь финиша",
    "Гимнасты, выступая в командном зачёте, поддерживали друг друга аплодисментами и советами, создавая атмосферу единства",
    "Бегунья, преодолевая барьер, чуть не упала, но сумела сохранить равновесие и продолжить забег, в итоге заняв призовое место",
    "На соревнованиях по стрельбе из лука участники концентрировались на мишени, игнорируя шум и давление со стороны зрителей",
    "Тренер, подводя итоги сезона, отметил прогресс каждого игрока и обозначил цели для дальнейшего развития команды.",
    "Искусственный интеллект, анализируя огромные массивы данных, способен выявлять закономерности, которые человек может не заметить даже за годы исследований",
    "Смартфоны нового поколения оснащены камерами с разрешением 200 мегапикселей, позволяющими делать снимки с невероятной детализацией даже в темноте",
    "В лаборатории учёные разрабатывают квантовые компьютеры, которые в будущем смогут решать задачи, недоступные современным суперкомпьютерам",
    "Система умного дома автоматически регулирует температуру, освещение и безопасность, адаптируясь к привычкам и потребностям жильцов",
    "Дроны, используемые для доставки грузов, сокращают время доставки и снижают затраты на логистику в труднодоступных районах",
    "Виртуальная реальность позволяет не только играть в игры, но и обучаться сложным профессиям, например, хирургии или пилотированию",
    "Нейросети, обученные на миллионах изображений, могут распознавать лица, предметы и даже эмоции с высокой точностью",
    "Блокчейн-технологии обеспечивают безопасность транзакций и защиту персональных данных, исключая возможность подделки или взлома",
    "Роботы-помощники в больницах доставляют лекарства и еду пациентам, освобождая медперсонал для более важных задач",
    "5G-сети обеспечивают скорость передачи данных в десятки раз выше, чем 4G, открывая новые возможности для интернета вещей и автономных транспортных средств",
    "Программисты создают алгоритмы, способные предсказывать погоду, эпидемии и экономические кризисы на основе анализа исторических данных",
    "3D-принтеры печатают не только пластиковые детали, но и биоматериалы, включая кожу и хрящи для медицинских операций",
    "Голосовые ассистенты понимают сложные запросы, поддерживают диалог и могут управлять целым домом с помощью одной команды",
    "Кибербезопасность становится всё важнее: хакеры используют ИИ для атак, а специалисты — для защиты цифровых систем",
    "Автономные автомобили, оснащённые лидарами и камерами, учатся безопасно передвигаться по городским улицам, избегая препятствий",
    "Облачные технологии позволяют хранить и обрабатывать данные в любом объёме, обеспечивая доступ к ним из любой точки мира",
    "Дополненная реальность накладывает цифровые объекты на реальный мир, помогая инженерам, врачам и учителям в их работе",
    "Нейроинтерфейсы позволяют управлять устройствами силой мысли, открывая перспективы для людей с ограниченными возможностями",
    "Интернет вещей объединяет бытовые приборы, транспорт и инфраструктуру в единую сеть, делая жизнь удобнее и эффективнее",
    "Разработчики создают метавселенные — виртуальные пространства, где люди могут работать, учиться и общаться, не выходя из дома",
    "Квантовая криптография обеспечивает абсолютно защищённую связь, которую невозможно взломать даже с помощью самых мощных компьютеров",
    "Роботизированные фабрики работают круглосуточно, минимизируя ошибки и увеличивая производительность в разы",
    "Алгоритмы машинного обучения помогают врачам ставить диагнозы, анализируя снимки МРТ и результаты анализов быстрее и точнее",
    "Энергоэффективные технологии снижают потребление ресурсов, способствуя устойчивому развитию и защите окружающей среды",
    "Технологии виртуальной примерки позволяют покупателям «надеть» одежду или аксессуары онлайн, прежде чем сделать заказ, экономя время и деньги"
]

In [5]:
import nltk
import pymorphy2
from string import punctuation
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize

def remove_stopwords(tokenized_texts):
    nltk.download('stopwords')
    stop_words = stopwords.words('russian')
    stop_words.extend(punctuation)

    clear_texts = []
    for words in tokenized_texts:
        clear_texts.append([word for word in words if word not in stop_words])
    return clear_texts

def lemmatize_text(tokenized_texts):
    lemmatizer = pymorphy2.MorphAnalyzer()
    lemmatized_data = []
    for words in tokenized_texts:
        lemmatized_data.append([lemmatizer.parse(word)[0].normal_form for word in words])
    return lemmatized_data

data_tok = [wordpunct_tokenize(sentence) for sentence in sentences]
data_tok = remove_stopwords(data_tok)
data_tok = lemmatize_text(data_tok)
data_tok[:2]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\TTPO100AJIEX\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


[['королева',
  'роскошный',
  'платье',
  'золотой',
  'вышивка',
  'торжественно',
  'войти',
  'тронный',
  'зал',
  'собраться',
  'придворный',
  'услышать',
  'её',
  'важный',
  'объявление'],
 ['наследный',
  'принц',
  'несмотря',
  'юный',
  'возраст',
  'проявлять',
  'мудрость',
  'хладнокровие',
  'необходимый',
  'управление',
  'огромный',
  'королевство',
  'неспокойный',
  'время']]

In [6]:
needed_tokens = set()
for sent in data_tok:
    needed_tokens.update(sent)

word2vec = {}
with open("model.txt", "r", encoding = "utf-8") as rf:
    next(rf)
    for line in rf:
        parts = line.rstrip().split(" ")
        token = '_'.join(parts[0].split('_')[:-1])
        if token in needed_tokens:
            word2vec[token] = np.asarray(parts[1:], dtype = float)

for key in needed_tokens:
    if key not in word2vec:
        print('Not found: ', key)

Not found:  её
Not found:  0
Not found:  «
Not found:  ия
Not found:  7
Not found:  нейроинтерфейс
Not found:  тёплый
Not found:  объём
Not found:  —
Not found:  далёкий
Not found:  часами
Not found:  лишь
Not found:  учёный
Not found:  2
Not found:  котёнок
Not found:  весёлый
Not found:  »
Not found:  пёс
Not found:  в
Not found:  плавец
Not found:  00
Not found:  счёт
Not found:  партнёр
Not found:  лидар
Not found:  метавселенная


In [7]:
from sklearn.cluster import KMeans

def sentence_embedding_mean(sentence) -> np.ndarray:
    vecs = [word2vec[word] for word in sentence if word in word2vec]
    return np.mean(vecs, axis = 0)

emb = np.array([ sentence_embedding_mean(sentence) for sentence in data_tok ])
kmeans = KMeans(n_clusters = 4, random_state = 42)
kmeans.fit(emb)
print('Dunn: ', dunn_score(emb, kmeans.labels_))

Dunn:  0.5787187547060944


Для улучшения будем использовать признаки tf/idf для каждого слова из словаря.

Для оценки качества будем использовтаь Dunn index. Не зря же мы его написали в прошлом задании :)

In [8]:
import math

def tf(word, sentence):
    occurences = { w: sentence.count(w) for w in word2vec }
    return occurences[word] / np.sum(list(occurences.values()))

def idf(word):
    cou = 0
    for sentence in data_tok:
        if sentence.count(word):
            cou += 1
    return 1 + math.log(len(data_tok) / cou)

def tfidf(word, sentence):
    return tf(word, sentence) * idf(word)
    
def sentence_embedding_tfidf(sentence) -> np.ndarray:
    return [ tfidf(word, sentence) for word in word2vec ]

In [9]:
import tqdm

emb = np.array([ sentence_embedding_tfidf(sentence) for sentence in tqdm.tqdm(data_tok) ])
kmeans = KMeans(n_clusters = 4, random_state = 42)
kmeans.fit(emb)
print('Dunn: ', dunn_score(emb, kmeans.labels_))

100%|██████████| 100/100 [00:15<00:00,  6.44it/s]


Dunn:  0.6938707014191031


Стало получше

Попробуем теперь "подмешать" к tfidf усредненные эмбеддинги слов; при усреднении будем использовать tfidf в качестве весов. Это позволит придать редким словам (которые, вероятно, имеют наибольшее значение для кластеризации) больший вес.

In [10]:
def sentence_embedding_combined(sentence) -> np.ndarray:
    vecs = [ word2vec[word] for word in sentence if word in word2vec]
    weights = np.array([tfidf(word, sentence) for word in sentence if word in word2vec])
    return [ tfidf(word, sentence) for word in word2vec ] + list(np.average(vecs, axis = 0, weights = weights))

In [11]:
emb = np.array([ sentence_embedding_combined(sentence) for sentence in tqdm.tqdm(data_tok) ])
kmeans = KMeans(n_clusters = 4, random_state = 42)
kmeans.fit(emb)
print('Dunn: ', dunn_score(emb, kmeans.labels_))

100%|██████████| 100/100 [00:15<00:00,  6.45it/s]


Dunn:  0.6576005890650894


Интересно, но стало чуть хуже. Тем не менее все равно гораздо лучше, чем простое усреднение